In [1]:
import os
import json
from datetime import datetime
from collections import Counter
import re

import importlib
import utility_descriptions
importlib.reload(utility_descriptions)
#############################


# =========================
# Tools
# =========================
from utility_descriptions import (
    generate_hollow_square_multi_flat_description,
    generate_hollow_square_multi_gable_description,
    generate_L_multi_flat_description,
    generate_L_multi_gable_description,
    generate_square_multi_flat_description,
    generate_square_multi_gable_description,
    generate_square_multi_hip_description,
    generate_square_single_flat_description,
    generate_square_single_gable_description,
    generate_square_single_hip_description,
    generate_T_multi_flat_description,
    generate_T_multi_gable_description,
    generate_U_multi_flat_description,
    generate_U_multi_gable_description
)

from utility_descriptions import (
    generate_construction_description,
    generate_space_description,
    generate_setpoint_description
)

from utility_descriptions import (
    generate_air_system_fcu_description,
    generate_air_system_vrf_description,
    generate_air_system_vav_description,
    generate_air_system_vav_fcu_descriptions,
    generate_air_system_dx_elec_description,
    generate_air_system_dx_fuel_description,
    generate_air_system_dx_hp_description,
    generate_air_system_doas_fcu_description,
    generate_air_system_doas_vrf_description
)

from utility_descriptions import (
    generate_chilled_water_description,
    generate_hot_water_description,
    generate_condenser_water_description
)

from user_prompts import USER_PROMPTS

from utility_system_prompts import (
    geo_system_prompt,
    info_system_prompt,
    air_system_prompt,
    water_system_prompt,
)

from openai import OpenAI


# =========================
# Configs
# =========================
MODEL_NAME = "qwen3-max"
NUM_SAMPLES_PER_PROMPT = 3 
SAVE_EVERY_N = 10
NGRAM_SIZE = 3  # n-gram value

OUT_PATH = "multi-modal_to_specific_description_Mar19.json"
# =========================

client = OpenAI(
    api_key="Your API", # your own api key
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1" # make sure the url is your url
)


def save_checkpoint(all_cases, failed_cases, out_path):

    tmp_path = out_path.replace(".json", "_tmp.json")
    payload = {
        "timestamp": datetime.now().isoformat(),
        "num_success": len(all_cases),
        "num_failed": len(failed_cases),
        "cases": all_cases,
        "failed_cases": failed_cases
    }
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, out_path)
    print(f"💾 Saved checkpoint -> {out_path} | success={len(all_cases)} failed={len(failed_cases)}", flush=True)


def load_checkpoint_if_exists(out_path):

    if not os.path.isfile(out_path):
        return [], []

    try:
        with open(out_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            all_cases = data
            failed_cases = []
        else:
            all_cases = data.get("cases", [])
            failed_cases = data.get("failed_cases", [])

        print(f"🔁 Resume detected: loaded {len(all_cases)} success + {len(failed_cases)} failed from {out_path}", flush=True)
        return all_cases, failed_cases

    except Exception as e:
        print(f"⚠️ Failed to load checkpoint ({out_path}): {type(e).__name__}: {e}", flush=True)
        print("⚠️ Will start from scratch to avoid blocking.", flush=True)
        return [], []


def call_llm(system_prompt: str, user_prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=False
    )
    return response.choices[0].message.content


def exec_and_join(code: str, results_var: str) -> str:

    locals()[results_var] = []
    wrapped = "\n".join(
        f"{results_var}.append({line})"
        for line in code.splitlines()
        if line.strip()
    )
    exec(wrapped)
    return "".join(locals()[results_var])

def text_to_ngram_counter(text: str, n: int = 3) -> Counter:

    text = (text or "").strip().lower()
    if not text:
        return Counter()
    if len(text) < n:
        return Counter({text: 1})
    return Counter(text[i:i+n] for i in range(len(text) - n + 1))


def cosine_similarity_counter(c1: Counter, c2: Counter) -> float:

    if not c1 and not c2:
        return 1.0
    if not c1 or not c2:
        return 0.0

    # dot product
    dot = 0.0

    if len(c1) > len(c2):
        c1, c2 = c2, c1
    for k, v in c1.items():
        dot += v * c2.get(k, 0)

    # norms
    norm1 = sum(v * v for v in c1.values()) ** 0.5
    norm2 = sum(v * v for v in c2.values()) ** 0.5
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
    return dot / (norm1 * norm2)


def compute_pairwise_cosine(results: list, n: int = 3) -> list[list[float]]:

    vecs = [text_to_ngram_counter(r, n) for r in results]
    size = len(results)
    matrix = [[0.0] * size for _ in range(size)]
    for i in range(size):
        matrix[i][i] = 1.0
        for j in range(i + 1, size):
            sim = cosine_similarity_counter(vecs[i], vecs[j])
            matrix[i][j] = matrix[j][i] = sim
    return matrix


def select_median_result(results: list, n: int = 3) -> str:

    if not results:
        return ""
    if len(results) == 1:
        return results[0]

    similarity_matrix = compute_pairwise_cosine(results, n)

    avg_sims = []
    for i in range(len(results)):
        sims = [similarity_matrix[i][j] for j in range(len(results)) if i != j]
        avg_sims.append(sum(sims) / len(sims) if sims else 0.0)

    best_idx = avg_sims.index(max(avg_sims))
    return results[best_idx]


def aggregate_stage_samples(samples: list, stage_name: str, ngram_n: int = 3) -> str:

    if not samples:
        print(f"  ⚠️ {stage_name}: No valid samples, returning empty", flush=True)
        return ""
    if len(samples) == 1:
        return samples[0]

    selected = select_median_result(samples, n=ngram_n)
    sims = []
    selected_vec = text_to_ngram_counter(selected, ngram_n)
    skipped_one_selected = False
    for s in samples:

        if (s == selected) and (not skipped_one_selected):
            skipped_one_selected = True
            continue
        sims.append(cosine_similarity_counter(selected_vec, text_to_ngram_counter(s, ngram_n)))

    avg_sim = sum(sims) / len(sims) if sims else 1.0
    print(f"  ✅ {stage_name}: aggregated {len(samples)} samples → selected (avg Cosine={avg_sim:.3f})", flush=True)
    return selected


# ============ main loop ============

def main():
    user_prompts = [
        v for k, v in sorted(
            USER_PROMPTS.items(),
            key=lambda x: int(x[0].split("_")[-1])
        )
    ]

    all_cases, failed_cases = load_checkpoint_if_exists(OUT_PATH)

    done_ids = set()
    for c in all_cases:
        cid = c.get("case_id")
        if isinstance(cid, int):
            done_ids.add(cid)
    for f in failed_cases:
        cid = f.get("case_id")
        if isinstance(cid, int):
            done_ids.add(cid)

    total = len(user_prompts)
    print(f"✅ Total prompts: {total}", flush=True)
    print(f"✅ Already done: {len(done_ids)}", flush=True)
    print(f"✅ Save every: {SAVE_EVERY_N}", flush=True)
    print(f"✅ Samples per prompt (for aggregation): {NUM_SAMPLES_PER_PROMPT}", flush=True)
    print(f"✅ N-gram size: {NGRAM_SIZE}", flush=True)

    newly_processed = 0

    for i, user_prompt in enumerate(user_prompts, 1):
        if i in done_ids:
            continue

        print(f"\n🔄 Processing Prompt {i} with {NUM_SAMPLES_PER_PROMPT} samples for aggregation\n", flush=True)
        
        geo_samples, info_samples, air_samples, water_samples = [], [], [], []
        stage_errors = []

        for sample_idx in range(NUM_SAMPLES_PER_PROMPT):
            try:
                print(f"  📦 Sample {sample_idx + 1}/{NUM_SAMPLES_PER_PROMPT}", flush=True)
                
                # (A) GEO
                code = call_llm(geo_system_prompt, user_prompt)
                geo_result = exec_and_join(code, "results_agent_1")
                geo_samples.append(geo_result)
                # print(geo_result)
                
                # (B) INFO
                code = call_llm(info_system_prompt, user_prompt)
                info_result = exec_and_join(code, "results_agent_2")
                info_samples.append(info_result)
                # print(info_result)

                # (C) AIR
                code = call_llm(air_system_prompt, user_prompt)
                air_result = exec_and_join(code, "results_agent_3")
                air_samples.append(air_result)
                # print(air_result)

                # (D) WATER
                code = call_llm(water_system_prompt, user_prompt)
                water_result = exec_and_join(code, "results_agent_4")
                water_samples.append(water_result)
                
            except Exception as e:
                print(f"  ⚠️ Sample {sample_idx + 1} failed: {type(e).__name__}: {e}", flush=True)
                stage_errors.append({
                    "sample_idx": sample_idx,
                    "error_type": type(e).__name__,
                    "error_msg": str(e)
                })

        if not (geo_samples or info_samples or air_samples or water_samples):
            failed_cases.append({
                "case_id": i,
                "user_prompt": user_prompt,
                "error_type": "AllSamplesFailed",
                "error_message": f"All {NUM_SAMPLES_PER_PROMPT} samples failed across all stages.",
                "stage_errors": stage_errors
            })
            print(f"❌ Prompt {i} failed: all samples failed. Skipping.\n", flush=True)
            newly_processed += 1
            if newly_processed % SAVE_EVERY_N == 0:
                save_checkpoint(all_cases, failed_cases, OUT_PATH)
            continue

        building_geo = aggregate_stage_samples(geo_samples, "GEO", ngram_n=NGRAM_SIZE)
        building_info = aggregate_stage_samples(info_samples, "INFO", ngram_n=NGRAM_SIZE)
        building_air = aggregate_stage_samples(air_samples, "AIR", ngram_n=NGRAM_SIZE)
        building_water = aggregate_stage_samples(water_samples, "WATER", ngram_n=NGRAM_SIZE)

        # merge & store
        air_str = json.dumps(building_air, ensure_ascii=False, indent=2)
        information_i = "\n\n".join([
            building_geo.strip(),
            building_info.strip(),
            air_str.strip(),
            building_water.strip() if isinstance(building_water, str) else str(building_water)
        ]).strip()

        case = {
            "case_id": i,
            "user_prompt": user_prompt,
            "building_geo": building_geo,
            "building_info": building_info,
            "building_air": building_air,
            "building_water": building_water,
            f"information_{i}": information_i,
            "aggregation_meta": {
                "num_samples_requested": NUM_SAMPLES_PER_PROMPT,
                "geo_valid_samples": len(geo_samples),
                "info_valid_samples": len(info_samples),
                "air_valid_samples": len(air_samples),
                "water_valid_samples": len(water_samples),
                "sample_errors": stage_errors,
                "ngram_size_used": NGRAM_SIZE
            }
        }

        all_cases.append(case)
        newly_processed += 1
        print(f"✅ Prompt {i} completed with aggregation.\n", flush=True)

        if newly_processed % SAVE_EVERY_N == 0:
            save_checkpoint(all_cases, failed_cases, OUT_PATH)

    save_checkpoint(all_cases, failed_cases, OUT_PATH)
    print("✅ All done.", flush=True)


if __name__ == "__main__":
    main()

✅ Total prompts: 29
✅ Already done: 0
✅ Save every: 10
✅ Samples per prompt (for aggregation): 3
✅ N-gram size: 3

🔄 Processing Prompt 1 with 3 samples for aggregation

  📦 Sample 1/3
  📦 Sample 2/3
  📦 Sample 3/3
  ✅ GEO: aggregated 3 samples → selected (avg Cosine=0.995)
  ✅ INFO: aggregated 3 samples → selected (avg Cosine=0.997)
  ✅ AIR: aggregated 3 samples → selected (avg Cosine=1.000)
  ✅ WATER: aggregated 3 samples → selected (avg Cosine=1.000)
✅ Prompt 1 completed with aggregation.


🔄 Processing Prompt 2 with 3 samples for aggregation

  📦 Sample 1/3
  📦 Sample 2/3
  📦 Sample 3/3
  ✅ GEO: aggregated 3 samples → selected (avg Cosine=1.000)
  ✅ INFO: aggregated 3 samples → selected (avg Cosine=1.000)
  ✅ AIR: aggregated 3 samples → selected (avg Cosine=1.000)
  ✅ WATER: aggregated 3 samples → selected (avg Cosine=1.000)
✅ Prompt 2 completed with aggregation.


🔄 Processing Prompt 3 with 3 samples for aggregation

  📦 Sample 1/3
  📦 Sample 2/3
  📦 Sample 3/3
  ✅ GEO: aggregated 